# Phase 2 — Monolingual IR Training (Kaggle)
**Datasets to attach before running:**
| Kaggle Dataset name | Role |
|---|---|
| `Phase1_Dataset` | LRL-IR source code + DatasetPhase2 parquets |
| `Phase1_2000` (or 4096 / 6000 / 8000) | Phase 1 distilled SentenceTransformer checkpoint |

## 1. Install dependencies

In [ ]:
%%capture
!pip install sentence-transformers rank-bm25 py-vncorenlp fastparquet POT python-dotenv scikit-learn

## 2. Paths & environment

Directory layout on Kaggle (from the dataset screenshot):
```
/kaggle/input/
  phase1-dataset/
    LRL-IR/              ← top-level folder inside the uploaded zip
      LRL-IR/            ← actual project root
        DatasetPhase2/
          1.parquet
          2.parquet
        components/
        models/
        utils/
        ...
  phase1-2000/           ← Phase 1 training output dataset
    kaggle/
      working/
        LRL-IR/
          sentence_transformer_multiling/   ← distilled student model
```

In [ ]:
import os, sys, torch

# ── Source code (read-only) ────────────────────────────────────────────────────
# The dataset 'Phase1_Dataset' was uploaded as a folder named 'LRL-IR',
# so Kaggle mounts it one level deeper: phase1-dataset/LRL-IR/LRL-IR/
PROJECT_DIR = "/kaggle/input/phase1-dataset/LRL-IR/LRL-IR/"

# ── Phase 1 distilled model (read-only) ───────────────────────────────────────
# Choose whichever Phase1_XXXX dataset you attached.
# The model folder is always at: kaggle/working/LRL-IR/sentence_transformer_multiling/
PHASE1_DATASET_NAME = "phase1-2000"   # ← change to phase1-4096 / phase1-6000 / phase1-8000 as needed
PHASE1_MODEL_DIR = f"/kaggle/input/{PHASE1_DATASET_NAME}/kaggle/working/LRL-IR/sentence_transformer_multiling/"

# ── Writable output ───────────────────────────────────────────────────────────
# Must end with '/' — the trainer concatenates paths with string concat, not os.path.join
OUTPUT_DIR = "/kaggle/working/"

DATASET_DIR = os.path.join(PROJECT_DIR, "DatasetPhase2")

sys.path.insert(0, PROJECT_DIR)
os.environ["PROJECT_DIR"] = OUTPUT_DIR  # trainer reads this env var for saving models

# Verify
print("PROJECT_DIR    :", PROJECT_DIR,    "→ exists:", os.path.isdir(PROJECT_DIR))
print("PHASE1_MODEL   :", PHASE1_MODEL_DIR, "→ exists:", os.path.isdir(PHASE1_MODEL_DIR))
print("DATASET_DIR    :", DATASET_DIR,    "→ exists:", os.path.isdir(DATASET_DIR))
print("OUTPUT_DIR     :", OUTPUT_DIR)

## 3. Training configuration

In [ ]:
# Use the Phase 1 distilled SentenceTransformer as the base model
if os.path.isdir(PHASE1_MODEL_DIR) and os.listdir(PHASE1_MODEL_DIR):
    PRETRAINED_MODEL = PHASE1_MODEL_DIR
    print(f"Base model: Phase 1 distilled model  →  {PRETRAINED_MODEL}")
else:
    # Fallback to the public HuggingFace model if Phase1 dataset is not attached
    PRETRAINED_MODEL = "vinai/phobert-base-v2"
    print(f"Phase1 model not found — falling back to HuggingFace: {PRETRAINED_MODEL}")

# ── Hyperparameters (Chapter 5 of the paper) ──────────────────────────────────
LANGUAGE        = "vi"    # Vietnamese
DO_MLM_FINETUNE = False   # True = MLM warm-up on the corpus first (~30 min extra)
CHUNK_LENGTH    = 128
BATCH_SIZE      = 32
MARGIN          = 1.0
LEARNING_RATE   = 1e-5
EPOCHS          = 4

# ── Iteration Configuration ────────────────────────────────────────────────
TOTAL_ITERATIONS  = 8
CURRENT_ITERATION = 1   # Change this from 1 to 8 across different Kaggle sessions

# If CURRENT_ITERATION > 1, you MUST attach the output dataset from the PREVIOUS iteration
# and provide the path to its saved model.pth here:
PREVIOUS_ITERATION_CHECKPOINT = ""
# Example: "/kaggle/input/phase2-iter1/custom_sentence_transformer_trained/vi/model.pth"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nDevice : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 4. Verify dataset

In [ ]:
import pandas as pd

parquet_files = sorted([f for f in os.listdir(DATASET_DIR) if f.endswith(".parquet")])
print(f"Parquet files: {parquet_files}")

frames = []
for pf in parquet_files:
    df = pd.read_parquet(os.path.join(DATASET_DIR, pf), engine="fastparquet")
    print(f"  {pf}: shape={df.shape}, columns={list(df.columns)}")
    frames.append(df)

full_df = pd.concat(frames, ignore_index=True)
full_df = full_df.drop_duplicates(subset=["id"]).reset_index(drop=True)
print(f"\nTotal unique documents: {len(full_df)}")

## 5. Split into document corpus & QD pairs

- `DocumentDataset` scans a dir for `.parquet` files and reads `id, url, title, text`
- `QueryDocDataset` scans a dir for `.parquet` files and reads `id, query`

Each goes into its own isolated directory.

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

DOC_CORPUS_DIR = os.path.join(OUTPUT_DIR, "doc_corpus")
PROC_DOC_DIR   = os.path.join(OUTPUT_DIR, "processed_docs")
QD_TEST_DIR    = os.path.join(OUTPUT_DIR, "qd_test")

for d in [DOC_CORPUS_DIR, PROC_DOC_DIR, QD_TEST_DIR]:
    os.makedirs(d, exist_ok=True)

# Document corpus — only the 4 columns DocumentDataset reads
full_df[["id", "url", "title", "text"]].to_parquet(
    os.path.join(DOC_CORPUS_DIR, "corpus.parquet"), engine="fastparquet", index=False)

# QD pairs — 80/20 split, seed=42 (matches the paper)
qd_df = full_df[["id", "query"]].copy()
train_qd, test_qd = train_test_split(qd_df, test_size=0.2, random_state=42)

# SPLIT TRAINING DATA INTO SUBSETS
subsets = np.array_split(train_qd, TOTAL_ITERATIONS)
for i, subset in enumerate(subsets, start=1):
    subset_dir = os.path.join(OUTPUT_DIR, f"qd_train_{i}")
    os.makedirs(subset_dir, exist_ok=True)
    subset.to_parquet(os.path.join(subset_dir, "qd_train.parquet"), engine="fastparquet", index=False)

test_qd.to_parquet(os.path.join(QD_TEST_DIR, "qd_test.parquet"), engine="fastparquet", index=False)

print(f"Document corpus : {len(full_df)} docs → {DOC_CORPUS_DIR}")
print(f"Train QD pairs  : {len(train_qd)} (split into {TOTAL_ITERATIONS} subsets)")
print(f"Test  QD pairs  : {len(test_qd)}")


## 6. Pre-create model save directories

`MonolingualRetrivalTrainer.train()` does string concatenation  
`os.getenv("PROJECT_DIR") + "sentence_transformer_finetune/vi"` — no `os.makedirs()` call inside —  
so we must create the directories here before training starts.

In [ ]:
ST_SAVE_PATH  = os.path.join(OUTPUT_DIR, f"sentence_transformer_finetune/{LANGUAGE}")
CST_SAVE_PATH = os.path.join(OUTPUT_DIR, f"custom_sentence_transformer_trained/{LANGUAGE}")

os.makedirs(ST_SAVE_PATH,  exist_ok=True)
os.makedirs(CST_SAVE_PATH, exist_ok=True)

print("SentenceTransformer will save to       :", ST_SAVE_PATH)
print("CustomSentenceTransformer will save to :", CST_SAVE_PATH)
print("Checkpoint (.pth) will be at           :", os.path.join(CST_SAVE_PATH, "model.pth"))

## 7. Train

In [ ]:
from models.monolingual_retrival_trainer import MonolingualRetrivalTrainer
import os

print(f"\n{'='*60}")
print(f"Starting EM Iteration {CURRENT_ITERATION}/{TOTAL_ITERATIONS} on subset qd_train_{CURRENT_ITERATION}")
print(f"{'='*60}")

checkpoint_path = PREVIOUS_ITERATION_CHECKPOINT if CURRENT_ITERATION > 1 else None

if CURRENT_ITERATION > 1 and not checkpoint_path:
    raise ValueError("You must provide PREVIOUS_ITERATION_CHECKPOINT when CURRENT_ITERATION > 1")

trainer = MonolingualRetrivalTrainer(
    document_dir                  = DOC_CORPUS_DIR,
    processed_doc_store_dir       = PROC_DOC_DIR,
    qd_dir                        = os.path.join(OUTPUT_DIR, f"qd_train_{CURRENT_ITERATION}"),
    pretrained_model_name_or_path = PRETRAINED_MODEL,
    checkpoint_path               = checkpoint_path,
    do_mlm_fine_tune              = DO_MLM_FINETUNE if CURRENT_ITERATION == 1 else False,
    language                      = LANGUAGE,
    chunk_length_limit            = CHUNK_LENGTH,
    device                        = DEVICE,
    batch_size                    = BATCH_SIZE,
    margin                        = MARGIN,
    learning_rate                 = LEARNING_RATE,
    epochs                        = EPOCHS,
)

sentence_transformer_path, custom_transformer_path = trainer.train()

print("\n" + "="*60)
print(f"Iteration {CURRENT_ITERATION} complete!")
print("Remember to save this Kaggle session output as a Dataset to use for the next iteration.")


## 8. Evaluate — MRR, Acc@1, Acc@5

In [ ]:
import json
from models.monolingual_retrival import MonoLingualRetrival

# Checkpoint path: trainer saves to {CST_SAVE_PATH}/model.pth
CHECKPOINT_PATH = os.path.join(CST_SAVE_PATH, "model.pth")

PROC_DOC_EVAL_DIR = os.path.join(OUTPUT_DIR, "processed_docs_eval")
os.makedirs(PROC_DOC_EVAL_DIR, exist_ok=True)

# MonoLingualRetrival expects the path to the model.pth FILE (not the directory)
retrieval_model = MonoLingualRetrival(
    document_dir          = DOC_CORPUS_DIR,
    processed_doc_store_dir = PROC_DOC_EVAL_DIR,
    custom_sentence_transformer_pretrained_or_save_path = CHECKPOINT_PATH,
    language              = LANGUAGE,
    original_query_doc_count  = 30,
    extended_query_doc_count  = 30,
    chunk_length_limit    = CHUNK_LENGTH,
    relevant_threshold    = 0.1,
    relevant_default_lowerbound = 0.25,
    device                = DEVICE,
    batch_size            = BATCH_SIZE,
)
retrieval_model.eval()
print("Retrieval model loaded for evaluation.")

In [ ]:
test_df = pd.read_parquet(os.path.join(QD_TEST_DIR, "qd_test.parquet"), engine="fastparquet")
print(f"Evaluating on {len(test_df)} test queries...")

mrr_sum = acc1_sum = acc5_sum = 0
details = []

for i, (_, row) in enumerate(test_df.iterrows()):
    query   = str(row["query"])
    # id is int64 in parquet; DocumentDataset writes it to JSON as int;
    # LexicalMatching returns it as int — compare as str to be safe
    true_id = str(row["id"])

    with torch.no_grad():
        doc_ids, scores = retrieval_model(query)

    ranked_ids = [str(d) for d, _ in
                  sorted(zip(doc_ids, scores), key=lambda x: x[1], reverse=True)]

    rr = 0.0
    for rank, rid in enumerate(ranked_ids, start=1):
        if rid == true_id:
            rr = 1.0 / rank
            break

    mrr_sum  += rr
    acc1_sum += 1 if ranked_ids and ranked_ids[0] == true_id else 0
    acc5_sum += 1 if true_id in ranked_ids[:5] else 0
    details.append({"query": query, "true_id": true_id,
                    "retrieved_top5": ranked_ids[:5], "rr": rr})

    if (i + 1) % 50 == 0:
        print(f"  [{i+1}/{len(test_df)}]  running MRR={mrr_sum/(i+1):.4f}")

n = len(test_df)
print("\n" + "="*40)
print(f"MRR   : {mrr_sum/n:.4f}")
print(f"Acc@1 : {acc1_sum/n:.4f}")
print(f"Acc@5 : {acc5_sum/n:.4f}")

result_path = os.path.join(OUTPUT_DIR, "eval_results.json")
with open(result_path, "w", encoding="utf-8") as f:
    json.dump({"MRR": mrr_sum/n, "Acc@1": acc1_sum/n, "Acc@5": acc5_sum/n,
               "details": details}, f, ensure_ascii=False, indent=2)
print(f"Results saved → {result_path}")

## 9. List output files

In [ ]:
for root, dirs, files in os.walk(OUTPUT_DIR):
    level  = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        size_mb = os.path.getsize(os.path.join(root, f)) / 1e6
        print(f"{indent}  {f}  ({size_mb:.2f} MB)")